# Data and Packages

In [1]:
# Main Packages 
import polars as pl
import numpy as np

# Clustering 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score

In [2]:
# Time constants
seconds_in_day = 60 * 60 * 24 
minutes_per_week = 7 * 24 * 60 
n_weeks = 8     
eight_weeks_seconds = n_weeks * minutes_per_week * 60

In [ ]:
# Range of k values: k = 2,...,8 
k_range = range(2, 9)

In [ ]:
# Load data, filter for human users & first 8 weeks of data
df = (
    pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz", has_header=False, separator=",",
                new_columns=['time','src_user','dest_user','src_comp','dest_comp',
                              'auth_type','logon_type','auth_orientation','outcome'])
    .filter(pl.col('src_user').str.starts_with('U'))
    .filter(pl.col('time') < eight_weeks_seconds)
    .collect(engine='streaming')
)

In [13]:
# Chosen features
feature_cols = [
    'log_n_events',
    'log_n_distinct_src',
    'log_n_distinct_dest',
    'failure_ratio',
    'c_bar',
    's_bar',
    ]

# Functions

In [14]:
# Build the features dataframe
def build_features(df, agg_minutes):

    agg_seconds = agg_minutes * 60

    return (
        df.lazy()
        .with_columns(
            bucket = pl.col('time') // agg_seconds,
            theta = ((pl.col('time') % seconds_in_day) / seconds_in_day) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user', 'bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
            log_n_events = pl.col('n_events').log(),
            log_n_distinct_src = pl.col('n_distinct_src').log(),
            log_n_distinct_dest = pl.col('n_distinct_dest').log(),
        ).collect()
        )

In [ ]:
# Slice the data to get the relevant week 
def cluster_preprocess(features_df, X_scaled, week, buckets_per_week):

    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    in_bin = features_df['bucket'].is_between(lb,ub).to_numpy()

    features_week = features_df.filter(in_bin)
    X_scaled_week = X_scaled[in_bin]

    return features_week, X_scaled_week 

In [ ]:
# Fit k-means & compute CH index
def fit_kmeans(features_df, X_scaled, week, k_range, buckets_per_week):

    _, X_scaled_week = cluster_preprocess(features_df, X_scaled, week, buckets_per_week)

    ch_scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=123, n_init=10)
        labels = km.fit_predict(X_scaled_week)
        ch_scores[k] = calinski_harabasz_score(X_scaled_week, labels)

    best_k = max(ch_scores, key=ch_scores.get)

    return week, ch_scores, best_k

In [ ]:
# Compute CH scores and best k for each week 
def ch_by_week(df, agg_hour_level, k_range):
    
    agg_minutes = round(agg_hour_level * 60)
    buckets_per_week = minutes_per_week // agg_minutes   

    features_df = build_features(df, agg_minutes)

    X = features_df.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    ch_scores_by_week = {}
    best_k_by_week = {}

    for week in range(1, n_weeks + 1):
        _, ch_scores, best_k = fit_kmeans(features_df, X_scaled, week, k_range, buckets_per_week)
        ch_scores_by_week[week] = ch_scores
        best_k_by_week[week] = best_k


    return ch_scores_by_week, best_k_by_week

In [ ]:
# Construct table of best k across aggregation levels and weeks
def best_k_across_levels(df, agg_hour_level, k_range = k_range):

    rows = []

    for agg_hour in agg_hour_level:
        _, best_k_by_week = ch_by_week(df, agg_hour, k_range)

        for week, best_k in best_k_by_week.items():
            rows.append({'agg_hours': agg_hour, 'week': week, 'best_k': best_k})

    return pl.DataFrame(rows)

# Optimal Cluster Sizes

In [ ]:
# Get best k across all aggregation levels and weeks
agg_levels = [1/60, 1/6, 0.5, 1, 2, 3, 4, 6, 8, 12, 24]  
results = best_k_across_levels(df, agg_levels)
k_table = results.pivot(on='agg_hours', index='week', values='best_k')

In [ ]:
# Display counts for each cluster size (optimal k)
values, counts = np.unique(k_table.drop('week').to_numpy(), return_counts=True)

for v, c in zip(values, counts):
    print(f"Cluster size: {v} -> Count: {c}")

Cluster size: 2 -> Count: 65
Cluster size: 3 -> Count: 9
Cluster size: 5 -> Count: 6
Cluster size: 6 -> Count: 8
